# 提示词缓存进阶（Prompt Caching 201）
> **注意：** 本指南仅适用于 GPT-5.6 之前的模型。对于 GPT-5.6 及更新的模型，请参阅[提示词缓存指南](https://developers.openai.com/api/docs/guides/prompt-caching)。

一份关于提示词缓存的实用指南：基础知识、性能影响、度量方法和优化策略。
## 1. 提示词缓存基础
模型提示词中经常包含重复的内容——例如系统指令、工具和消息。当某个请求包含系统最近处理过的前缀时，OpenAI 可以将其路由到已经计算过该前缀的服务器，让模型复用之前的计算结果，而不必从头重新计算。
提示词缓存最多可将首 token 延迟（time-to-first-token）降低 80%，并将输入 token 成本最多降低 90%。它会在所有 API 请求上自动生效，并且不收取额外费用。
本 cookbook 的目标是更深入地探讨如何优化缓存命中。请先阅读我们的 [API 文档](https://developers.openai.com/api/docs/guides/prompt-caching)和[提示词缓存入门（Prompt Caching 101）](https://developers.openai.com/cookbook/examples/prompt_caching101/)以获得全面的概览——下面我们就开始吧！
### 1.1 基础知识
- 缓存命中需要完全一致的重复前缀匹配，并且对包含 1024 个或更多 token 的提示词自动生效，缓存命中以 128 个 token 为增量。
- 整个请求前缀都可以被缓存：消息、图像、音频、工具定义以及结构化输出 schema。
- 内存中的提示词缓存会在你所有的 API 请求上自动生效。扩展提示词缓存（Extended prompt caching）可将保留时间延长到 24 小时。
- 只有当两个请求共享相同的前缀并落到同一台机器上时，缓存才会生效。对于共享公共前缀的流量，可以利用可选参数 `prompt_cache_key` 来改善路由。
- 仔细考虑压缩（compaction）等上下文工程技术对缓存的影响。
- 在迭代过程中，通过请求日志或 Usage 仪表盘监控缓存、成本和延迟。

## 2. 为什么缓存很重要
### 2.1 核心技术原因：跳过预填充（Prefill）计算
在输入 token 上经过 transformer 各层的前向传播，是推理成本和延迟的主要来源。在 transformer 堆栈中，提示词缓存专门作用于注意力层内部的 key 和 value 投影。注意力是一种机制，它让模型在处理某个特定 token 时，能够衡量输入序列中不同部分的重要性，从而捕捉上下文和长距离依赖关系。

当你向 OpenAI 发出请求时，我们会创建 token 嵌入，然后将其转换为三个向量：一个 query（Q），表示该 token 为了理解自身上下文需要什么；一个 key（K），编码每个 token 代表什么；以及一个 value（V），包含在相关时可以被纳入的信息。模型将当前 token 的 query 与所有 token 的 key 进行比较（通过点积），得到注意力分数。经过 softmax 之后，这些分数成为权重，决定每个 token 的 value 对更新后的表示贡献多少。例如，在短语“she sat by the river bank”中，单词“river”为“bank”提供了重要的上下文，使其含义偏向“河岸”而不是“银行”。KV 缓存存储了该前缀在所有层和所有注意力头上的 key 和 value 张量。

如果你接着发出另一个请求——“she sat by the river bank on a cloudy day”——前缀的前半部分是完全相同的。在处理新的、未见过的后缀时，模型会复用缓存的张量，只为新 token 计算注意力。

关键在于，“river”和“bank”之间强烈的语义关系，已经在之前的前向传播中被编码进了这些缓存的 key 和 value 表示中。模型无需重新计算“bank”与“river”是如何对应的。相反，“cloudy”和“day”等新 token 会生成自己的 query，并对前缀中缓存的 key 进行注意力计算。换句话说，语义基础已经打好；模型只需在已有的注意力状态之上继续构建，而不必从头重新计算。

### 2.2 成本影响
缓存折扣可能相当可观。折扣幅度因模型系列而异——随着我们的推理堆栈变得越来越高效，最新的模型能够提供更大的缓存折扣。下面是一些示例，所有模型的价格请参阅我们的定价页面。所有近期的模型（gpt-4o 及更新的模型）都启用了提示词缓存。
| 模型 | 输入 <br>（每 1M token） | 缓存输入 <br>（每 1M token） | 缓存折扣 |
| --- | --- | --- | --- |
| gpt-4o | $2.50 | $1.25 | 50.00% |
| gpt-4.1 | $2.00 | $0.50 | 75.00% |
| gpt-5-nano | $0.05 | $0.005 | 90.00% |
| gpt-5.2 | $1.75 | $0.175 | 90.00% |
| gpt-realtime（音频） | $32.00 | $0.40 | 98.75% |


### 2.3 延迟影响
降低首 token 延迟（TTFT）是提高缓存率的主要动机之一。缓存的 token 最多可将延迟降低约 80%。
缓存使延迟大致与生成的输出长度成正比，而不是与整个对话长度成正比，因为完整的历史上下文无需重新预填充。当缓存命中时，模型采样的开销是线性的，而不是二次方的。
我将一系列提示词运行了 2300 次，并绘制了缓存请求与未缓存请求的对比图。对于最短的提示词（1024 个 token），缓存请求快 7%；而在较长的一端（150k+ token），TTFT 快了 67%。输入越长，缓存对首 token 延迟的影响就越大。

![图 2](../images/prompt-caching-201/figure-2.svg)

## 3. 先度量缓存（以便迭代）
### 3.1 单个请求：`cached_tokens`
响应中包含 usage 字段，用于表明有多少提示词 token 是由缓存提供的。所有请求都会在 [Response](https://developers.openai.com/api/docs/api-reference/responses/object) 或 [Chat](https://developers.openai.com/api/docs/api-reference/chat/object) 对象的 `usage.prompt_tokens_details` 中显示一个 `cached_tokens` 字段，表示有多少提示词 token 被缓存。


```
"usage": {
  "prompt_tokens": 2006,
  "completion_tokens": 300,
  "total_tokens": 2306,
  "prompt_tokens_details": {
    "cached_tokens": 1920
  },
  "completion_tokens_details": {
    "reasoning_tokens": 0,
    "accepted_prediction_tokens": 0,
    "rejected_prediction_tokens": 0
  }
}
```



你还可以在 usage 仪表盘中按度量筛选缓存/未缓存的 token，从而获得整体概览

![图 3](../images/prompt-caching-201/figure-3.png)

## 4. 提高缓存命中率（实战手册）

什么是缓存命中？就是当一个请求以与之前某个请求相同的前缀开头时，系统可以复用之前计算好的 key/value 张量，而不必重新计算。提高缓存命中率——尽可能多地复用前缀——会直接提升性能和效率，因此最大化缓存命中率就是我们的目标！


### 4.1 发送超过 1024 个 token 的提示词
这可能有些违反直觉，但在某些情况下，让提示词稍微长一点反而能降低总成本。
假设你有一个 900 个 token 的提示词——你永远不会得到缓存命中。如果你把提示词加长到 1100 个 token，并获得 50% 的缓存率，就能节省 33% 的 token 成本。如果缓存率达到 70%，则能节省 55%。一旦跨过缓存阈值并实现有意义的复用，这些重复 token 的边际成本就会大幅下降。这意味着一个稍长但稳定的前缀，可能比一个永远无法被缓存的较短提示词更便宜。

### 4.2 稳定前缀
这是投入最少、效果最大的优化：保持提示词前面的部分稳定。
将持久不变的内容放在开头：
- 指令
- 工具定义
- Schema

将易变的内容（例如用户输入、动态值、会话特定的数据）移到末尾。即使前面的 token 发生很小的变化，也会破坏精确的前缀匹配，导致无法命中缓存。

**来自 Codex 的经验**
当我们的工程团队介绍他们[如何设计 Codex 智能体循环](https://openai.com/index/unrolling-the-codex-agent-loop/)时，他们强调将提示词结构视为一等的性能要素，并把缓存作为首要优先级。在 Codex CLI 中，系统指令、工具定义、沙箱配置和环境上下文在各次请求之间保持完全相同且顺序一致，以保留长而稳定的提示词前缀。当运行时配置在对话中途发生变化时（例如新的工作目录或审批模式），智能体循环会追加新消息，而不是修改之前的消息。通过避免更改原始前缀，系统保留了精确的前缀匹配，而这正是提示词缓存命中所必需的。

**提示：使用 metadata**
我们见过一些客户为了日后查找/调试，在请求的靠前位置加入时间戳，结果意外地使缓存失效。把它移到 `metadata` 中，就不会影响缓存了！

### 4.3 保持工具和 Schema 完全一致
工具、schema 及其顺序都属于缓存前缀的一部分——它们会被注入到开发者指令之前，这意味着修改它们会使缓存失效。这包括：
- Schema 键的变化
- 工具顺序的变化
- 指令的变化


**提示：在不破坏提示词缓存的情况下调整工具**
利用 [`allowed_tools`](https://developers.openai.com/api/docs/guides/function-calling/#tool-choice) 这个 `tool_choice` 选项。它允许你在单次请求中限制模型可以调用的工具，而无需修改 tools 数组、导致缓存失效。在 tools 中列出你的完整工具集，然后使用 `allowed_tools` 块指定在某一轮中可以使用哪些工具。


```
# Static “tools” stay in the cached prompt prefix: 

tools = [get_weather_def, get_location_def, calendar_def, …] 

# Per-call “allowed_tools” only lives in request metadata, not in the prefix: 

allowed_tools = {"mode":"auto", "tools":["get_weather","get_location"]}
```




### 4.4 使用 `prompt_cache_key` 提高路由粘性
只有当两个请求共享相同的前缀并落到同一台机器上时，缓存才会生效。请求会根据提示词前约 256 个 token 的哈希值被路由到推理引擎。当你提供 `prompt_cache_key` 时，它会与该哈希值组合，以提高路由粘性——也就是说，具有相同前缀的请求更有可能落到同一个引擎上，并复用缓存的 KV 状态。这种方法很有效——我们的一位编程类客户在开始使用 `prompt_cache_key` 后，命中率从 60% 提高到了 87%。

当你发送的不同请求具有相同的初始上下文（即前 256 个 token）、但后面的内容各不相同时，`prompt_cache_key` 尤其有用，因为它能让你有意识地将相关请求分组。

对于每个前缀 + `prompt_cache_key` 组合，推理引擎每分钟大约能处理 ~15 个请求。如果流量超过这个速率——例如，你发送了数千个共享相同前缀和键的请求——系统会将多出的请求分配到其他机器上以实现负载均衡。每台新机器都会产生一次性的缓存未命中。

这意味着你应该选择合适的键粒度，使每个前缀 + 键组合保持在约 15 RPM 以下。

在编程类用例中，我们观察到：
- 按用户设置的键能提高相关对话之间的复用（例如在同一个代码库中工作）。
- 当用户并行运行许多不相关的会话时，按对话设置的键扩展性更好。
将多个用户分组共享一个键也是一种不错的做法。一个简单的算法是：对用户 ID 进行哈希，再按“桶”的数量取模，使每个桶的请求速率接近 15 RPM，从而最大化缓存性能。


**提示：用 Flex Processing 代替 Batch API 进行测试**
如果你的工作流对延迟不敏感，你可能正在使用 Batch API。若想在缓存方面获得更多灵活性，可以考虑使用 Flex Processing。Flex 提供与 Batch 相同的 50% token 折扣，但通过 Responses API 运行，只需在每个请求中指定 `service_tier="flex"`。这为你提供了更多灵活性：

- 控制请求速率（RPM）
- 可以使用扩展提示词缓存
- 可以加入 prompt_cache_key

由于你可以更精确地调整路由和缓存局部性，Flex 在某些工作负载中可以实现更高的缓存命中率。它特别适合那些推理负载不高、但仍能从成本优化中受益的原型开发或生产工作负载。

我用 10,000 个相同的请求进行了一次正面对比测试。与 Batch 作业相比，Flex（使用扩展提示词缓存和 `prompt_cache_key`）的缓存命中率提高了 8.5%。这一改进带来了 23% 的输入 token 成本降低。

![图 4](../images/prompt-caching-201/figure-4.svg)

Batch API 在缓存方面并非支持所有模型——GPT-5 之前的模型不受支持，因此如果你正在使用 o3 或 o4-mini，应考虑切换到 Flex 以利用缓存。请在我们的[定价文档](https://developers.openai.com/api/docs/pricing?latest-pricing=flex)中查看这方面的最新信息。


**洞察：把 `prompt_cache_key` 看作分片键**
在思考我们在后端如何路由请求时，把 `prompt_cache_key` 看作数据库分片键可能会有所帮助——优化该参数时的考量非常相似。与分片键一样，粒度是一种权衡。对于给定的前缀，每台机器大约只能处理 ~15 RPM，因此如果你在太多请求上使用同一个键，请求就会溢出到多台机器上。每到一台新机器，缓存都要重新开始。另一方面，如果你的键过于细粒度，流量会分散到各台机器上，你就会失去缓存复用的好处。路由仍然是负载均衡的——`prompt_cache_key` 提高了相似提示词命中同一服务器的概率，但并不保证粘性——缓存始终是尽力而为的！

### 4.5 使用 Responses API 而不是 Chat Completions
正如我们在[我们为什么构建 Responses API](https://developers.openai.com/blog/responses-api/) 中所述，我们的内部基准测试显示，与 Chat Completions 相比，请求的缓存利用率提高了 40-80%。
这是因为与 Chat Completions 不同，Responses API 会通过 `previous_response_id`（如果你采用无状态方式，则通过[加密推理项](https://platform.openai.com/docs/guides/reasoning?api-mode=responses#encrypted-reasoning-items)）在各轮之间保留原始思维链 token。Chat Completions 不提供保留这些 token 的方式。如果你没有使用推理模型，则不会有缓存方面的改进。[使用 Responses API 提升推理模型的性能](https://developers.openai.com/cookbook/examples/responses_api/reasoning_items#caching)是一份深入理解这一点的优秀指南。

### 4.6 审慎对待上下文工程
上下文工程的核心，是决定每次请求中模型的输入包含哪些内容。每个模型都有固定的上下文窗口，但精心挑选每次请求传入的内容，不仅仅是为了不超出限制。随着输入不断增长，你不仅在逼近截断，还在增加重放成本，并迫使模型将注意力分散到更多的 token 上。有效的上下文工程，就是有意识地管理这一约束的实践。
然而，当你丢弃、总结或压缩对话中较早的轮次时，就会破坏缓存。从这个角度看，上下文工程与提示词缓存天然存在矛盾——一个优先考虑动态性，另一个优先考虑稳定性。

随着运行时间更长的智能体和原生压缩工具的兴起，在设计架构时牢记缓存非常重要，以确保在成本与智能之间取得合适的平衡。


```
response = client.responses.create(
        model="gpt-5.2-codex",
        input=conversation,
        store=False,
        context_management=[{"type": "compaction", "compact_threshold": 100000}],
    )
```



一种常见的失败模式是工具集过于臃肿，导致应该调用哪个工具变得模糊不清。保持精简的工具集可以提高决策质量，也有利于长期上下文。在按请求挑选工具时，请参考提示 4.3，利用 [`allowed_tools`](https://developers.openai.com/api/docs/guides/function-calling/#tool-choice) 这个 `tool_choice` 选项进行裁剪。


**实用法则**
利用你的评估（evals）来选择压缩方法和频率，在成本（既包括通过截断/总结减少总输入 token，也包括缓存）与精心的上下文工程所带来的智能提升之间取得平衡

## 5. 故障排查：为什么缓存率可能偏低
常见原因：
- 工具或响应格式 schema 发生变化
- 因触及模型的上下文窗口而进行的简单截断
- 指令或系统提示词发生变化
- 推理强度（reasoning effort）发生变化
- 缓存过期：间隔时间过长，保存的前缀已被丢弃。
- 加入了空格、时间戳或其他动态内容
- 在推理模型上使用 Chat Completions，因为隐藏的思维链 token 会被丢弃

## 6. 扩展提示词缓存与零数据保留

[扩展提示词缓存](https://developers.openai.com/api/docs/guides/prompt-caching#extended-prompt-cache-retention)的工作方式是：当内存已满时，将 key/value 张量转移到 GPU 本地存储中，从而显著增加可用于缓存的存储容量。对于 `gpt-5.5`、`gpt-5.5-pro` 以及所有未来的模型，默认值为 `24h`，并且不支持 `in_memory`。

```
{
  "model": "gpt-5.1",
  "input": "Write me a haiku about froge...",
  "prompt_cache_retention": "24h"
}
```



KV 张量是模型注意力层在预填充阶段产生的中间表示。只有 key/value 张量可能会被持久化到本地存储中；所有原始输入内容都只保留在内存中。

**洞察：实际缓存的是什么？**
KV 缓存只保存模型的 key/value 张量（隐藏状态的线性投影），以便我们在下一个推理步骤中复用它们。KV 缓存是一种中间表示。它本质上只是模型内部的一堆数字——这意味着无论采用何种保留策略，都不会存储任何原始文本/多模态输入。

## 7. Realtime API

Realtime API 中的缓存与 Responses API 的工作方式相同——传入的所有音频、文本或图像都会被缓存，前缀的任何变化都会破坏缓存。不过，由于 Realtime API 的上下文窗口较短（目前为 32k），管理截断就显得尤为重要。一个上下文为 32k、最大输出 token 为 4,096 的模型，在发生截断之前，上下文中只能容纳 28,224 个 token。

### 7.1 保留比例（Retention Ratio）
默认情况下 `(truncation: "auto")`，服务器只会移除刚好足够多的旧消息，以使内容适配上下文窗口。一旦超出限制，这种“即时”裁剪会在每一轮都让对话的起始位置略微后移。然而，这种简单的策略会导致频繁的缓存未命中。
[`retention_ratio`](https://developers.openai.com/api/docs/guides/realtime-costs/#truncation) 设置通过让你控制保留与丢弃多少早期上下文来改变这一点。它是一种可配置的截断策略，让开发者能够控制保留多少 token 上下文窗口，在保留上下文与优化缓存命中之间取得平衡。
例如，`retention_ratio: 0.7` 意味着发生截断时，系统会保留现有对话窗口的大约 70%，并丢弃最旧的约 30%。丢弃会在一次较大的截断事件中完成（而不是一点点地逐步移除）。


```
{
  "event": "session.update",
  "session": {
    "truncation": {
      "type": "retention_ratio",
      "retention_ratio": 0.7
    }
  }
}
```



这会产生一个更稳定、能跨越多轮保持不变的前缀，从而减少反复的缓存失效。
其代价是截断以更大的块进行，因此你可能会一次性丢失更多的对话历史。这意味着与逐步截断相比，模型可能会更早地突然“忘记”对话的早期部分。实际上，你是在用记忆深度换取缓存稳定性（更一致的前缀）。
下图展示了简单的逐轮截断方式与使用 `retention_ratio` 时的效果对比。

![图 5](../images/prompt-caching-201/figure-5.svg)


## 结论
提示词缓存是 OpenAI 平台上杠杆效应最高的优化手段之一。当你的前缀保持稳定、路由设计合理时，就可以在不改变模型行为或质量的前提下，显著降低成本和延迟。
关键思想简单而有力：稳定前缀、监控 `cached_tokens`、注意约 15 RPM 的限制，并审慎地使用 `prompt_cache_key`。对于高并发量或对延迟不敏感的工作负载，可以考虑使用 Flex Processing 和扩展保留，以进一步提升缓存局部性。
缓存是尽力而为且与路由相关的，但只要有意识地进行设计，它就能带来显著的效率提升。像对待任何其他性能系统一样对待它：先度量，再有针对性地迭代，并在设计提示词时就考虑复用。